# D-03. 진동 세기와 떨어진 미등록 점수 — 특성 MLP 임베딩

D-02 결과: 프로토타입 거리가 진동 세기를 따라감 → 조용한 미등록 재질(carpet·soft·wood)을 놓침 (AUROC 0.33~0.40)

새 점수 (모두 클수록 미등록)

| 이름 | 방법 | 노리는 것 |
|---|---|---|
| `scaled` | 재질별 거리 ÷ 그 재질 학습 구간의 거리 중앙값, 최솟값 | 좁게 뭉친 조용한 재질 주변의 거리를 크게 |
| `scaled_nn` | 원래 거리로 가장 가까운 재질을 고른 뒤 그 재질 퍼짐으로만 나눔 | `scaled`에서 넓은 재질이 흡수하는 문제 피하기 |
| `cmaha` | 재질별 공분산(Ledoit-Wolf) Mahalanobis | 재질마다 다른 퍼짐 모양 반영 |
| `ratio` | 가장 가까운 ÷ 두 번째 가까운 프로토타입 거리 | 거리 크기와 무관하게 "애매함" |

기존 `proto`, `maha`, `knn`, `msp`, `energy`와 같은 147 split에서 비교하고, **점수와 진동 세기의 상관(`rho_vib`)이 줄었는지** 함께 본다

실행: D-01과 같은 준비(임베딩 zip을 MyDrive/에), CPU로 충분

## 0. 환경 준비

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = "D/ood-mlp"   # main에 merge된 뒤에는 "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/ML-imu").exists():
        !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
    os.chdir("/content/ML-imu")
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
    !pip install -q -r requirements.txt
    !python data/download.py

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path += [str(ROOT / "src"), str(ROOT / "notebooks" / "D_ood")]
print(ROOT)

In [ ]:
# 임베딩 위치: 받은 zip을 내 드라이브 맨 위(MyDrive/)에 올려 두면 Colab 안에 풀어서 쓴다
# (드라이브에 풀린 폴더 MyDrive/ML-imu_embeddings가 있으면 그 폴더를 바로 쓴다)
EMB_ZIP = "ML-imu_embeddings*.zip"

if IN_COLAB:
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    OUT = MYDRIVE / "ML-imu_D"
    EMB = MYDRIVE / "ML-imu_embeddings"
    if not EMB.exists():
        EMB = ROOT / "embeddings" / "mlp"
        if not any(EMB.glob("emb_mlp_*.npy")):
            zips = sorted(MYDRIVE.glob(EMB_ZIP))
            assert zips, f"MyDrive/에 {EMB_ZIP} 파일을 올려 주세요"
            EMB.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zips[-1]) as zf:
                for info in zf.infolist():
                    if not info.is_dir():
                        (EMB / Path(info.filename).name).write_bytes(zf.read(info))
            print("압축 해제:", zips[-1].name)
else:
    EMB = ROOT / "embeddings" / "mlp"
    OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

n_emb = len(list(EMB.glob("emb_mlp_*.npy")))
print(EMB, n_emb, "개 임베딩 (기대값 168 = 147 + 21)")

## 1. 데이터

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recordings import load_dataset, CHANNEL_NAMES
from splits import load_material_folds
from evaluate import log_result
from ood import run_heldout, EMBEDDING_SCORERS, SCALE_FREE_SCORERS, DIFFICULTY

X, y, rec = load_dataset("data")
folds = load_material_folds()
vib = X[:, :, CHANNEL_NAMES.index("acc_z")].std(axis=1)   # 구간별 진동 세기

QUIET = ["carpet", "soft_pvc", "soft_tiles", "wood"]       # 진동이 조용한 그룹 (정리 문서 3.2 ③)
order = QUIET + ["fine_concrete", "concrete", "hard_tiles", "tiled"]
print(X.shape, len(folds), "개 split")

## 2. 147 split 실행 (방법 9개)

In [ ]:
MODEL = "mlp"
SCORERS = {**EMBEDDING_SCORERS, **SCALE_FREE_SCORERS}
TAU_METHODS = ("proto", "scaled", "scaled_nn", "cmaha", "ratio")

def load(held_out, fold):
    tag = f"{MODEL}_{held_out}_{fold}"
    return np.load(EMB / f"emb_{tag}.npy"), np.load(EMB / f"logits_{tag}.npy")

df = run_heldout(load, y, rec, model=MODEL, folds=folds,
                 scorers=SCORERS, tau_methods=TAU_METHODS, vib=vib)
df["group"] = np.where(df.heldout.isin(QUIET), "quiet", "loud")
df.to_csv(OUT / f"D_scalefree_{MODEL}_folds.csv", index=False)
print(len(df), "행 저장")

## 3. 결과 표

In [ ]:
METHODS = ["proto", "scaled", "scaled_nn", "cmaha", "ratio", "maha", "knn", "msp", "energy"]

summary = (df.groupby("method")
           .agg(auroc=("auroc", "mean"), fpr95=("fpr95", "mean"), rho_vib=("rho_vib", "mean"))
           .join(df.pivot_table(index="method", columns="group", values="auroc").add_prefix("auroc_"))
           .loc[METHODS])
print("방법별 평균 (rho_vib: 점수와 진동 세기의 Spearman 상관, 0에 가까울수록 진동과 무관)")
display(summary.round(3))

auroc_tab = df.pivot_table(index="heldout", columns="method", values="auroc").loc[order, METHODS]
print("빠진 재질별 AUROC")
display(auroc_tab.round(3))

## 4. τ 판정

- `known_pass`: 알려진 재질 테스트 구간 통과율 (목표 0.95)
- `unknown_reject`: 빠진 재질 구간을 미등록으로 판정한 비율

In [ ]:
tau_tab = (df[df.method.isin(TAU_METHODS)]
           .pivot_table(index="heldout", columns="method", values=["known_pass", "unknown_reject"])
           .loc[order])
tau_tab.loc["평균"] = tau_tab.mean()
display(tau_tab.round(3))

## 5. 그림: 조용한 / 덜컹거리는 재질별 AUROC와 진동 상관

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
summary[["auroc_quiet", "auroc_loud"]].plot.bar(ax=axes[0], width=0.8)
axes[0].axhline(0.5, color="gray", ls="--", lw=1)
axes[0].set_ylabel("AUROC"); axes[0].set_ylim(0, 1); axes[0].set_title("held-out group: quiet vs loud")
axes[0].legend(["quiet (carpet, soft, wood)", "loud (concrete, tiles)"], loc="upper center",
               bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
summary["rho_vib"].plot.bar(ax=axes[1], color="tab:gray")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_ylabel("Spearman(score, vibration)"); axes[1].set_title("dependence on vibration")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(OUT / f"D_scalefree_{MODEL}.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. results.csv 기록 (새 방법만)

기존 5개 방법은 D-01에서 이미 기록함

In [ ]:
new = df[df.method.isin(SCALE_FREE_SCORERS)]
rows = new.groupby(["method", "heldout"])[["auroc", "fpr95"]].mean().reset_index()
for _, r in rows.iterrows():
    log_result("D", f"{MODEL}+{r.method}", {"auroc": r.auroc, "fpr95": r.fpr95},
               setting=f"heldout={r.heldout}", fold="mean",
               note=f"{DIFFICULTY.get(r.heldout, 'mid')}, scale-free (D-03)")

import shutil
shutil.copy(ROOT / "results" / "results.csv", OUT / "results_with_D03.csv")
print(len(rows), "줄 기록")

## 메모할 것

- `rho_vib`가 proto보다 줄어든 방법:
- 조용한 재질 AUROC가 0.5를 넘은 방법:
- 덜컹거리는 재질 성능을 유지했는지:
- τ에서 미등록 거부율: